ModuleNotFoundError: No module named 'pgmpy'

  Using cached click-8.5.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached idna-3.20-py3-none-any.whl.metadata (7.2 kB)
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 2.4/2.4 MB 15.5 MB/s  0:00:00
   ---------------------------------------- 0.0/839.9 kB ? eta -:--:--
   ---------------------------------------- 0.0/839.9 kB ? eta -:--:--
   ---------------------------------------- 0.0/839.9 kB ? eta -:--:--
   ---------------------------------------- 0.0/839.9 kB ? eta -:--:--
   ---------------------------------------- 839.9/839.9 kB 5.7 MB/s  0:00:00
Using cached click-8.5.0-py3-none-any.whl (125 kB)
   ---------------------------------------- 0.0/3.8 MB ? eta -:--:--
   -------- ------------------------------- 0.8/3.8 MB 10.7 MB/s eta 0:00:01
   -------------------------------- ------- 3.1/3.8 MB 16.7 MB/s eta 0:00:01
   ---------------------------------------- 3.8/3.8 MB 10.3 MB/s  0:00:00
Using cached idna-3.20-py3

In [4]:
import pandas as pd
import seaborn as sns

from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.inference import VariableElimination

# Load Titanic dataset
df = sns.load_dataset('titanic')
df = df[['survived', 'pclass', 'sex', 'age']]
df.dropna(inplace=True)

# Discretize age into bins
df['age'] = pd.cut(
    df['age'],
    bins=[0, 12, 30, 50, 100],
    labels=['child', 'young', 'adult', 'senior']
)

# Convert categorical columns to numeric codes
df['sex'] = df['sex'].astype('category').cat.codes
df['age'] = df['age'].astype('category').cat.codes

# Define Bayesian Network
model = DiscreteBayesianNetwork([
    ('pclass', 'survived'),
    ('sex', 'survived'),
    ('age', 'survived')
])

# Train the model
model.fit(df)

# Create inference object
inference = VariableElimination(model)

# Query 1
result1 = inference.query(
    variables=['survived'],
    evidence={'pclass': 1, 'sex': 0, 'age': 1}
)
print("Query 1 - 1st class, female, young:")
print(result1)

# Query 2
result2 = inference.query(
    variables=['survived'],
    evidence={'pclass': 3, 'sex': 1, 'age': 2}
)
print("\nQuery 2 - 3rd class, male, adult:")
print(result2)

# Query 3
result3 = inference.query(
    variables=['survived'],
    evidence={'pclass': 2, 'sex': 1, 'age': 3}
)
print("\nQuery 3 - 2nd class, male, senior:")
print(result3)

# Query 4
result4 = inference.query(
    variables=['survived'],
    evidence={'pclass': 1, 'sex': 0, 'age': 0}
)
print("\nQuery 4 - 1st class, female, child:")
print(result4)

C:\Users\IZY\anaconda3\Lib\site-packages\pgmpy\estimators\__init__.py:4: FutureWarning: `pgmpy.estimators.StructureScore` is deprecated and will be removed in v1.3.0. Use `pgmpy.structure_score` instead.
  from .StructureScore import (


Query 1 - 1st class, female, young:
+-------------+-----------------+
| survived    |   phi(survived) |
+=============+=================+
| survived(0) |          0.0294 |
+-------------+-----------------+
| survived(1) |          0.9706 |
+-------------+-----------------+

Query 2 - 3rd class, male, adult:
+-------------+-----------------+
| survived    |   phi(survived) |
+=============+=================+
| survived(0) |          0.8732 |
+-------------+-----------------+
| survived(1) |          0.1268 |
+-------------+-----------------+

Query 3 - 2nd class, male, senior:
+-------------+-----------------+
| survived    |   phi(survived) |
+=============+=================+
| survived(0) |          0.9167 |
+-------------+-----------------+
| survived(1) |          0.0833 |
+-------------+-----------------+

Query 4 - 1st class, female, child:
+-------------+-----------------+
| survived    |   phi(survived) |
+=============+=================+
| survived(0) |          1.0000 |
+-----